# Task 2: Laning & Overtaking with SB3 PPO

This notebook trains a PPO agent on the newer `highway-env` / Stable-Baselines3 API, following the older Task 2 setup from `racetrack-agents` where three slower non-agent vehicles are spawned and the ego vehicle must lane-follow while overtaking.

The older DQN command used `--spawn_vehicles 3`, `--batch_size 256`, `--lr 0.00005`, `--lr_decay`, `--arch Identity`, and `--fc_layers 3`. The cells below map those ideas to SB3 PPO with a 3-layer MLP policy, linear learning-rate decay, and `other_vehicles=3` in the `racetrack-oval-v0` config.

In [1]:
# If this notebook is running in a fresh environment, install the core packages first.
# In the local repo environment you can usually leave this cell commented out.
#
# %pip install "highway-env>=1.8" "stable-baselines3[extra]>=2.0" tensorboard moviepy

from pathlib import Path
from copy import deepcopy
import base64
import os
import platform
import random
import subprocess
import sys
import time

import gymnasium as gym
from gymnasium.wrappers import RecordVideo
import highway_env  # Registers highway-env environments in many versions.
import numpy as np
import torch.nn as nn
import torch

from IPython.display import HTML, display
from stable_baselines3 import PPO
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.callbacks import BaseCallback, CheckpointCallback, EvalCallback
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv

# Newer gymnasium versions can register an external environment package explicitly.
# Older highway-env versions register on import, so we keep this tolerant.
try:
    gym.register_envs(highway_env)
except Exception:
    pass


c:\Users\16469\anaconda3\envs\circuit\lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


## Experiment Config

Task 2 is represented by `other_vehicles=3`. The notebook now starts from `RacetrackEnvOval.default_config()` and overrides only the pieces that define this experiment, so future highway-env API changes are easier to absorb.

In [2]:
SEED = 42
ENV_ID = "racetrack-v0"

# Keep full training as the default. For an end-to-end notebook smoke test, run
# `FAST_DEV_RUN=1` in the process environment before executing the notebook.
FAST_DEV_RUN = True
USE_SUBPROC = False

# Prefer CPU in notebooks unless a CUDA-enabled GPU is available and stable.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

EXP_ID = "sb3_ppo_custom_cnn"
if FAST_DEV_RUN:
    EXP_ID += "_fastdev"

WORK_DIR = Path.cwd()

RUN_DIR = WORK_DIR / "runs" / EXP_ID
MODEL_DIR = RUN_DIR / "models"
BEST_MODEL_DIR = MODEL_DIR / "best"
CHECKPOINT_DIR = MODEL_DIR / "checkpoints"
LOG_DIR = RUN_DIR / "logs"
VIDEO_DIR = RUN_DIR / "videos"
TB_LOG_DIR = RUN_DIR / "tensorboard"

for directory in [MODEL_DIR, BEST_MODEL_DIR, CHECKPOINT_DIR, LOG_DIR, VIDEO_DIR, TB_LOG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Reproducibility: exact runs can still vary across machines/GPU kernels.
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

Using device: cuda


In [3]:
from highway_env.envs import racetrack_env
from race_env import RacetrackFast
racetrack_env.RacetrackFast = RacetrackFast
gym.register(id=ENV_ID, entry_point="race_env:RacetrackFast")

c:\Users\16469\anaconda3\envs\circuit\lib\site-packages\gymnasium\envs\registration.py:694: UserWarning: WARN: Overriding environment racetrack-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")


In [4]:
# Fast mode now performs a real short PPO run with multiple rollouts/evaluations so progress is visible.
FAST_DEV_TIMESTEPS = int(os.environ.get("FAST_DEV_TIMESTEPS", "8192"))

# Full mode keeps the older command's 5000-episode intent, using the current API's episode horizon.
N_EPISODES = 5000
TOTAL_TIMESTEPS = FAST_DEV_TIMESTEPS if FAST_DEV_RUN else N_EPISODES * RacetrackFast.default_config["duration"]

# Use one env in notebooks on Windows; full mode can use several vectorized workers.
N_ENVS = 1 if FAST_DEV_RUN else min(8, max(1, os.cpu_count() or 1))

# PPO minibatch size: smaller in fast mode, older command value in full mode.
BATCH_SIZE = 128 if FAST_DEV_RUN else 512

# Rollout length per env before each PPO update; longer rollouts stabilize PPO on this task.
N_STEPS = 1024 if FAST_DEV_RUN else 2048

# Starting learning rate, mapped from the older `--lr 0.00005` setting but slightly higher for quicker learning.
LEARNING_RATE = 2.5e-4 #5e-5

# Number of SGD passes per PPO update; fewer in fast mode keeps iteration time reasonable.
N_EPOCHS = 6 if FAST_DEV_RUN else 10

# Evaluation cadence in environment steps; fast mode evaluates often so you can see progress.
EVAL_FREQ = 2048 if FAST_DEV_RUN else max(10_000 // N_ENVS, 1)

# Evaluation episodes per callback; three is enough to see trend during a quick run.
N_EVAL_EPISODES = 3 if FAST_DEV_RUN else 5

# A small entropy bonus is helpful with richer occupancy features and throttle control.
ENT_COEF = 0.001

# Slightly smaller clip range helps learning stay stable when the action space grows.
CLIP_RANGE = 0.15

# Stronger discounting and GAE smoothing can improve horizon-aware lane-following and overtaking behavior.
GAMMA = 0.99
GAE_LAMBDA = 0.97
MAX_GRAD_NORM = 0.5


## Build Training and Evaluation Environments

SB3 trains on vectorized environments. `DummyVecEnv` is the safest default inside notebooks on Windows. For longer command-line runs, set `USE_SUBPROC = True`.

In [5]:
def make_task2_env(render_mode=None):
    """Create one Task 2 racetrack environment."""
    return gym.make(ENV_ID, config=RacetrackFast.default_config(), render_mode=render_mode)

vec_env_cls = SubprocVecEnv if USE_SUBPROC and N_ENVS > 1 else DummyVecEnv

train_env = make_vec_env(
    lambda: make_task2_env(),
    n_envs=N_ENVS,
    seed=SEED,
    vec_env_cls=vec_env_cls,
)
# Evaluation stays single-env so callback results are easy to interpret.
eval_env = make_vec_env(
    lambda: make_task2_env(),
    n_envs=1,
    seed=SEED + 10_000,
    vec_env_cls=DummyVecEnv,
)


## Define PPO

The PPO policy uses a 3-layer actor and critic MLP, matching the spirit of `--arch Identity --fc_layers 3` from the older code: flatten the occupancy grid, then learn dense policy/value heads. Comments below explain every model setting that differs from SB3 defaults or maps to the older command.

In [6]:
class RacetrackCNN(BaseFeaturesExtractor):
    """
    Lightweight CNN for the 11×12×12 OccupancyGrid.
    Input:  [batch, 11, 12, 12]
    Output: flat feature vector of size features_dim
    """
    def __init__(self, observation_space, features_dim=256):
        super().__init__(observation_space, features_dim)
        n_channels = observation_space.shape[0]  # 11
        self.cnn = nn.Sequential(
            nn.Conv2d(n_channels, 32, kernel_size=3, padding=1),  # → [32, 12, 12]
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),           # → [64, 12, 12]
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=2),            # → [64, 5, 5]
            nn.ReLU(),
            nn.Flatten(),                                           # → 1600
            nn.Linear(1600, features_dim),
            nn.ReLU(),
        )

    def forward(self, obs):
        return self.cnn(obs.float())

policy_kwargs = dict(
    features_extractor_class=RacetrackCNN,
    features_extractor_kwargs=dict(features_dim=256),
    net_arch=dict(pi=[128, 64], vf=[128, 64]),  # separate actor/critic heads
)

In [7]:
def linear_schedule(initial_value):
    """SB3 schedule: progress_remaining moves from 1.0 to 0.0 during training."""
    def schedule(progress_remaining):
        return progress_remaining * initial_value
    return schedule

model = PPO(
    policy="CnnPolicy",  # Flattened occupancy-grid input, equivalent in spirit to the older Identity backbone.
    env=train_env,  # Vectorized Task 2 racetrack environment.
    learning_rate=linear_schedule(LEARNING_RATE),  # Implements the older `--lr_decay` behavior.
    n_steps=N_STEPS,  # Rollout length before each PPO update.
    batch_size=BATCH_SIZE,  # Minibatch size for PPO optimization.
    n_epochs=N_EPOCHS,  # Number of optimization passes over each rollout buffer.
    gamma=GAMMA,  # Discount factor tuned for longer-horizon lane-following and overtaking.
    gae_lambda=GAE_LAMBDA,  # GAE smoothing tuned for richer rewards.
    clip_range=CLIP_RANGE,  # Slightly smaller clip range improves stability with richer observations.
    ent_coef=ENT_COEF,  # Small entropy bonus to keep exploration alive during overtaking.
    max_grad_norm=MAX_GRAD_NORM,  # Gradient clipping for stable policy updates.
    policy_kwargs=policy_kwargs,  # Actor/critic architecture defined above.
    tensorboard_log=str(TB_LOG_DIR),  # Training curves and eval metrics.
    seed=SEED,  # Reproducible initialization and rollout seeds where supported.
    verbose=1,  # Print rollout/evaluation progress in notebook output.
    device=DEVICE,
)

model.policy


Using cuda device


ActorCriticCnnPolicy(
  (features_extractor): RacetrackCNN(
    (cnn): Sequential(
      (0): Conv2d(10, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU()
      (4): Conv2d(64, 64, kernel_size=(3, 3), stride=(2, 2))
      (5): ReLU()
      (6): Flatten(start_dim=1, end_dim=-1)
      (7): Linear(in_features=1600, out_features=256, bias=True)
      (8): ReLU()
    )
  )
  (pi_features_extractor): RacetrackCNN(
    (cnn): Sequential(
      (0): Conv2d(10, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU()
      (4): Conv2d(64, 64, kernel_size=(3, 3), stride=(2, 2))
      (5): ReLU()
      (6): Flatten(start_dim=1, end_dim=-1)
      (7): Linear(in_features=1600, out_features=256, bias=True)
      (8): ReLU()
    )
  )
  (vf_features_extractor): RacetrackCNN(


## Train and Save the Best Model

`EvalCallback` periodically runs deterministic evaluations and writes the best model to disk. TensorBoard logs are stored under `runs/sb3_ppo_task2_laning_overtaking/logs`.

In [8]:
# Start TensorBoard in the notebook while training is running.
# This opens the log directory in a background server and prints the local URL.
#
%load_ext tensorboard
%tensorboard --logdir TB_LOG_DIR

# If the magic above is not available, you can also launch a standalone server from a terminal:
# tensorboard --logdir "runs/sb3_ppo_laning_overtaking_richocc_throttle_fastdev/tensorboard" --host 127.0.0.1 --port 6006


Reusing TensorBoard on port 6006 (pid 59216), started 2 days, 12:18:43 ago. (Use '!kill 59216' to kill it.)

In [11]:
class EarlyStoppingCallback(BaseCallback):
    def __init__(self, check_freq=10000, patience=3, min_delta=0.0, verbose=0):
        super().__init__(verbose)
        self.check_freq = check_freq
        self.patience = patience
        self.min_delta = min_delta
        self.best_mean_reward = -np.inf
        self.epochs_without_improvement = 0

    def _on_step(self) -> bool:
        if self.n_calls % self.check_freq == 0:
            # Use the current training reward estimate from the rollout buffer.
            # This is intentionally conservative and does not require a full eval loop.
            current_mean_reward = np.mean(self.locals.get("rewards", [0.0]))
            if current_mean_reward > self.best_mean_reward + self.min_delta:
                self.best_mean_reward = current_mean_reward
                self.epochs_without_improvement = 0
            else:
                self.epochs_without_improvement += 1
            if self.epochs_without_improvement >= self.patience:
                if self.verbose:
                    print(f"Early stopping at step {self.num_timesteps} with no improvement.")
                return False
        return True


early_stop_callback = EarlyStoppingCallback(check_freq=max(5_000 // N_ENVS, 1), patience=2, min_delta=0.1, verbose=1)

eval_callback = EvalCallback(
    eval_env,
    best_model_save_path=str(BEST_MODEL_DIR),
    log_path=str(LOG_DIR / "eval"),
    eval_freq=EVAL_FREQ,
    n_eval_episodes=N_EVAL_EPISODES,
    deterministic=True,
    render=False,
)

checkpoint_callback = CheckpointCallback(
    save_freq=max(50_000 // N_ENVS, 1),
    save_path=str(CHECKPOINT_DIR),
    name_prefix="ppo_checkpoint",
)

In [ ]:
model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=[early_stop_callback, eval_callback, checkpoint_callback],
    tb_log_name="PPO",
)

Logging to c:\Users\16469\Desktop\circuit\racetrack-agents\runs\sb3_ppo_custom_cnn_fastdev\tensorboard\PPO_1
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 5.56     |
|    ep_rew_mean     | 2.47     |
| time/              |          |
|    fps             | 15       |
|    iterations      | 1        |
|    time_elapsed    | 65       |
|    total_timesteps | 1024     |
---------------------------------
Eval num_timesteps=2048, episode_reward=6.79 +/- 0.00
Episode length: 12.00 +/- 0.00
------------------------------------------
| eval/                   |              |
|    mean_ep_length       | 12           |
|    mean_reward          | 6.79         |
| time/                   |              |
|    total_timesteps      | 2048         |
| train/                  |              |
|    approx_kl            | 0.0041474877 |
|    clip_fraction        | 0.0605       |
|    clip_range           | 0.15         |
|    entropy_loss         | -2.83   

In [ ]:
model.save(MODEL_DIR / "ppo_last")
train_env.close()
eval_env.close()

In [15]:
import importlib

for module_name in ["onnx", "onnxruntime"]:
    try:
        importlib.import_module(module_name)
    except ModuleNotFoundError:
        os.system('pip install --quiet "onnx==1.12.0" "onnxruntime==1.12.0"')

import onnx
import onnxruntime as ort

In [30]:
best_model_path = BEST_MODEL_DIR / "best_model.zip"
last_model_path = MODEL_DIR / "ppo_task2_last.zip"

trained_model = PPO.load(
    best_model_path if best_model_path.exists() else last_model_path,
    device="cpu",
)
trained_model.policy.eval()
trained_model.policy.to("cpu")

obs_shape = train_env.observation_space.shape      # e.g. (11, 12, 12)
dummy_obs  = torch.zeros(1, *obs_shape, dtype=torch.float32)  # [1,C,H,W]

print(f"Observation space : {obs_shape}")
print(f"Dummy input shape : {tuple(dummy_obs.shape)}")

Observation space : (10, 12, 12)
Dummy input shape : (1, 10, 12, 12)


In [25]:
class ActorOnlyWrapper(nn.Module):
    """
    obs → action_mean
    Deterministic forward pass: features → mlp_pi → action_net.
    No sampling, no value head.  Use this in RE Engine.
    """
    def __init__(self, policy):
        super().__init__()
        self.policy = policy

    def forward(self, obs: torch.Tensor) -> torch.Tensor:
        # obs arrives as [batch, C, H, W] — 4D, no reshaping needed.
        features          = self.policy.features_extractor(obs.float())
        latent_pi, _      = self.policy.mlp_extractor(features)
        return self.policy.action_net(latent_pi)   # [batch, 2]

In [26]:
class FullPolicyWrapper(nn.Module):
    """
    obs → (action_mean, value)
    Exports both actor and critic heads.
    Useful for debugging / distillation; not needed for RE Engine inference.
    """
    def __init__(self, policy):
        super().__init__()
        self.policy = policy

    def forward(self, obs: torch.Tensor):
        features               = self.policy.features_extractor(obs.float())
        latent_pi, latent_vf   = self.policy.mlp_extractor(features)
        action_mean            = self.policy.action_net(latent_pi)   # [batch, 2]
        value                  = self.policy.value_net(latent_vf)    # [batch, 1]
        return action_mean, value

In [27]:
def export_and_verify(
    wrapper:      nn.Module,
    dummy_input:  torch.Tensor,
    out_path:     Path,
    output_names: list[str],
    opset:        int = 17,
) -> ort.InferenceSession:

    wrapper.eval()
    out_path.parent.mkdir(parents=True, exist_ok=True)

    # Sanity-check forward pass before tracing
    with torch.no_grad():
        test_out = wrapper(dummy_input)
    if isinstance(test_out, tuple):
        for i, t in enumerate(test_out):
            print(f"  pre-export output[{i}]: {tuple(t.shape)}")
    else:
        print(f"  pre-export output: {tuple(test_out.shape)}")

    # ONNX trace
    with torch.no_grad():
        torch.onnx.export(
            wrapper,
            dummy_input,
            str(out_path),
            export_params       = True,
            opset_version       = opset,
            do_constant_folding = True,
            input_names         = ["obs"],
            output_names        = output_names,
            dynamic_axes        = {
                "obs": {0: "batch_size"},
                **{name: {0: "batch_size"} for name in output_names},
            },
            dynamo = False,
        )

    # ONNX structural check
    onnx_model = onnx.load(str(out_path))
    onnx.checker.check_model(onnx_model)

    # Print graph I/O summary
    print(f"\n✅  {out_path.name}")
    for node in onnx_model.graph.input:
        dims = [d.dim_value if d.dim_value else d.dim_param
                for d in node.type.tensor_type.shape.dim]
        print(f"   input  '{node.name}' : {dims}")
    for node in onnx_model.graph.output:
        dims = [d.dim_value if d.dim_value else d.dim_param
                for d in node.type.tensor_type.shape.dim]
        print(f"   output '{node.name}' : {dims}")

    # OnnxRuntime inference check
    sess = ort.InferenceSession(str(out_path), providers=["CPUExecutionProvider"])
    dummy_np  = dummy_input.numpy()
    ort_outs  = sess.run(None, {"obs": dummy_np})
    for out_meta, arr in zip(sess.get_outputs(), ort_outs):
        print(f"   ORT  '{out_meta.name}' : shape={arr.shape}  "
              f"min={arr.min():.4f}  max={arr.max():.4f}")

    return sess

In [28]:
ACTOR_PATH = MODEL_DIR / "ppo_actor_only.onnx"
FULL_PATH  = MODEL_DIR / "ppo_full_policy.onnx"

In [31]:
print("=" * 60)
print("Exporting  actor-only  ONNX …")
actor_session = export_and_verify(
    ActorOnlyWrapper(trained_model.policy),
    dummy_obs,
    ACTOR_PATH,
    output_names=["action_mean"],
)

Exporting  actor-only  ONNX …
  pre-export output: (1, 2)


C:\Users\16469\AppData\Local\Temp\ipykernel_41292\2718528617.py:23: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(



✅  ppo_actor_only.onnx
   input  'obs' : ['batch_size', 10, 12, 12]
   output 'action_mean' : ['batch_size', 2]
   ORT  'action_mean' : shape=(1, 2)  min=-0.0095  max=-0.0089


In [32]:
print()
print("=" * 60)
print("Exporting  full policy  ONNX …")
full_session = export_and_verify(
    FullPolicyWrapper(trained_model.policy),
    dummy_obs,
    FULL_PATH,
    output_names=["action_mean", "value"],
)


Exporting  full policy  ONNX …
  pre-export output[0]: (1, 2)
  pre-export output[1]: (1, 1)

✅  ppo_full_policy.onnx
   input  'obs' : ['batch_size', 10, 12, 12]
   output 'action_mean' : ['batch_size', 2]
   output 'value' : ['batch_size', 1]
   ORT  'action_mean' : shape=(1, 2)  min=-0.0095  max=-0.0089
   ORT  'value' : shape=(1, 1)  min=0.2675  max=0.2675


C:\Users\16469\AppData\Local\Temp\ipykernel_41292\2718528617.py:23: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


In [33]:
C, H, W = obs_shape
print(f"""
╔══════════════════════════════════════════════════════════╗
║             ONNX Export Summary — C# Reference           ║
╠══════════════════════════════════════════════════════════╣
║  File (RE Engine)   : ppo_actor_only.onnx                ║
║  Input  "obs"       : [1, {C}, {H}, {W}]  (NCHW)              ║
║  Output "action_mean": [1, 2]                            ║
║     action[0] = throttle ∈ [-1,1] → map to accel m/s²   ║
║     action[1] = steering ∈ [-1,1] → map to angle rad     ║
╠══════════════════════════════════════════════════════════╣
║  File (debug only)  : ppo_full_policy.onnx               ║
║  Input  "obs"       : [1, {C}, {H}, {W}]  (NCHW)              ║
║  Output "action_mean": [1, 2]                            ║
║  Output "value"     : [1, 1]  (critic estimate)          ║
╚══════════════════════════════════════════════════════════╝
""".format(C=C, H=H, W=W))


╔══════════════════════════════════════════════════════════╗
║             ONNX Export Summary — C# Reference           ║
╠══════════════════════════════════════════════════════════╣
║  File (RE Engine)   : ppo_actor_only.onnx                ║
║  Input  "obs"       : [1, 10, 12, 12]  (NCHW)              ║
║  Output "action_mean": [1, 2]                            ║
║     action[0] = throttle ∈ [-1,1] → map to accel m/s²   ║
║     action[1] = steering ∈ [-1,1] → map to angle rad     ║
╠══════════════════════════════════════════════════════════╣
║  File (debug only)  : ppo_full_policy.onnx               ║
║  Input  "obs"       : [1, 10, 12, 12]  (NCHW)              ║
║  Output "action_mean": [1, 2]                            ║
║  Output "value"     : [1, 1]  (critic estimate)          ║
╚══════════════════════════════════════════════════════════╝



## Load the Best PPO Checkpoint

If training was interrupted before an evaluation improved, fall back to the last saved model.

In [34]:
best_model_path = BEST_MODEL_DIR / "best_model.zip"
last_model_path = MODEL_DIR / "ppo_last.zip"

if best_model_path.exists():
    trained_model = PPO.load(best_model_path)
    print(f"Loaded best model: {best_model_path}")
else:
    trained_model = PPO.load(last_model_path)
    print(f"Best model was not found, loaded last model: {last_model_path}")


Loaded best model: c:\Users\16469\Desktop\circuit\racetrack-agents\runs\sb3_ppo_custom_cnn_fastdev\models\best\best_model.zip


## Find and Export the Best Evaluation Episode

The first pass evaluates deterministic rollouts over fixed seeds without recording. The best seed is then replayed once with `RecordVideo`, producing a single video for the strongest episode found in this sweep.

In [35]:
def run_episode(model, seed, record=False, name_prefix="ppo_cnn_best_episode"):
    """Run one deterministic episode and optionally record it to VIDEO_DIR."""
    render_mode = "rgb_array" if record else None
    env = gym.make(ENV_ID, config=RacetrackFast.default_config(), render_mode=render_mode)

    if record:
        env = RecordVideo(
            env,
            video_folder=str(VIDEO_DIR),
            name_prefix=name_prefix,
            episode_trigger=lambda episode_id: episode_id == 0,
        )
    obs, info = env.reset(seed=seed)
    done = False
    truncated = False
    total_reward = 0.0
    episode_length = 0

    while not (done or truncated):
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, truncated, info = env.step(action)
        total_reward += float(reward)
        episode_length += 1
        if record:
            env.render()

    env.close()
    return total_reward, episode_length

candidate_seeds = list(range(SEED, SEED + (1 if FAST_DEV_RUN else 25)))
episode_scores = []

for seed in candidate_seeds:
    reward, length = run_episode(trained_model, seed=seed, record=False)
    episode_scores.append({"seed": seed, "reward": reward, "length": length})

best_episode = max(episode_scores, key=lambda item: item["reward"])
best_episode


{'seed': 42, 'reward': 464.4214398507333, 'length': 751}

In [36]:
video_prefix = f"ppo_task2_best_seed_{best_episode['seed']}"
recorded_reward, recorded_length = run_episode(
    trained_model,
    seed=best_episode["seed"],
    record=True,
    name_prefix=video_prefix,
)

video_files = sorted(VIDEO_DIR.glob(f"{video_prefix}*.mp4"), key=lambda path: path.stat().st_mtime)
best_video_path = video_files[-1] if video_files else None

print(f"Recorded reward: {recorded_reward:.3f}")
print(f"Recorded length: {recorded_length}")
print(f"Video path: {best_video_path}")


c:\Users\16469\anaconda3\envs\circuit\lib\site-packages\gymnasium\wrappers\record_video.py:94: UserWarning: WARN: Overwriting existing videos at c:\Users\16469\Desktop\circuit\racetrack-agents\runs\sb3_ppo_custom_cnn_fastdev\videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


MoviePy - Building video c:\Users\16469\Desktop\circuit\racetrack-agents\runs\sb3_ppo_custom_cnn_fastdev\videos\ppo_task2_best_seed_42-episode-0.mp4.
MoviePy - Writing video c:\Users\16469\Desktop\circuit\racetrack-agents\runs\sb3_ppo_custom_cnn_fastdev\videos\ppo_task2_best_seed_42-episode-0.mp4



MoviePy - Done !
MoviePy - video ready c:\Users\16469\Desktop\circuit\racetrack-agents\runs\sb3_ppo_custom_cnn_fastdev\videos\ppo_task2_best_seed_42-episode-0.mp4
Recorded reward: 464.421
Recorded length: 751
Video path: c:\Users\16469\Desktop\circuit\racetrack-agents\runs\sb3_ppo_custom_cnn_fastdev\videos\ppo_task2_best_seed_42-episode-0.mp4


## Display the Exported Video

In [37]:
def show_video(video_path, width=720):
    """Embed an exported mp4 directly in the notebook."""
    video_path = Path(video_path)
    video_bytes = video_path.read_bytes()
    encoded = base64.b64encode(video_bytes).decode("ascii")
    display(HTML(f"""
    <video width="{width}" controls>
      <source src="data:video/mp4;base64,{encoded}" type="video/mp4">
    </video>
    """))

if best_video_path is not None:
    show_video(best_video_path)
else:
    print("No video file was found. Check that moviepy/ffmpeg are installed and rerun the recording cell.")


## Optional: TensorBoard

Run this cell while training or after training to inspect reward, loss, entropy, KL, and evaluation curves.

In [ ]:
# Uncomment these lines in an interactive notebook session.

